# Part 5 — Fix model saving and continue the managed pipeline

ใช้ WSL/kernel `itcs355-lab3` เดิม รัน code cell เดียว

สาเหตุที่ยืนยันแล้ว: MLflow/skops บล็อก `sklearn.tree._tree.Tree` ขณะ save โมเดล
ใช้ cloudpickle แบบเดียวกับ Lab 2 สำหรับโมเดลที่เราฝึกเอง ไม่เปลี่ยน seed/model/gate

cell นี้จะ:
- ตรวจ log เดิมและ resources โดยไม่ทำ setup/role changes
- commit เฉพาะ source ที่ allowlist ในเครื่อง ไม่มี push; เก็บ state/source เก่าไว้
- ส่ง normal-repair job ชื่อคงที่ รันซ้ำ cell จะติดตาม job ชื่อนี้ ไม่สร้าง attempt ใหม่
- รัน DAG หกขั้นใหม่เพื่อให้ได้หลักฐานครบ (งานเดิม force_rerun=True จึง reuse ไม่ได้)
- ฝึก normal candidate ใหม่ด้วยวิธี save ที่แก้แล้ว แล้ว evaluate/register/deploy ตาม gate เดิม
- หากล้มเหลว ดึง child error/log ในการรันนี้ทันที

**มีค่าใช้จ่าย compute และ deployment ตามการใช้งานจริง**
ไม่ฝึก degraded ซ้ำ; ไม่มีการเปลี่ยนสิทธิ์หรือสร้าง compute ใหม่
หากสำเร็จ staging จะยังเปิดอยู่ ต้องเก็บหลักฐาน/cleanup ต่อ
ส่ง SEND THIS SUMMARY พร้อม error ด้านบนถ้ามี ไม่ต้องกลับไป run 05/05b ใหม่


In [1]:
from pathlib import Path
import platform
import subprocess
import sys

if platform.system() != "Linux" or "microsoft" not in platform.release().lower():
    raise RuntimeError("เลือก kernel itcs355-lab3 ฝั่ง WSL ก่อนครับ")
here = Path.cwd().resolve()
lab = next((base / "lab5" for base in (here, *here.parents)
            if (base / "lab5/scripts/repair_serialization.py").is_file()
            and (base / "lab4/README.md").is_file()), None)
if lab is None:
    raise RuntimeError("เปิด notebook จาก repository เดิมก่อนครับ")
print("Repair model saving — local source commit + one managed normal-repair job; no setup.", flush=True)
with subprocess.Popen(
    [sys.executable, "-u", "-m", "scripts.repair_serialization"], cwd=lab,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
) as task:
    try:
        for line in task.stdout:
            print(line, end="", flush=True)
        exit_code = task.wait()
    except KeyboardInterrupt:
        task.terminate()
        try:
            task.wait(timeout=10)
        except subprocess.TimeoutExpired:
            task.kill()
        print("Azure job may continue. Inspect Studio; do not start a duplicate.", flush=True)
        raise
if exit_code:
    raise RuntimeError("Normal repair หยุด: ส่ง summary และ child error ด้านบน ไม่ต้องเริ่มใหม่ครับ")
print("ส่ง SEND THIS SUMMARY กลับมาครับ staging ยังเปิดอยู่และต้อง cleanup ต่อ", flush=True)


Repair model saving — local source commit + one managed normal-repair job; no setup.
[15:34:25] 1/3 Check the inspected failure and existing resources; no setup
[15:34:42] 2/3 Commit the serialization fix locally and snapshot revised source; no push
[15:34:43] Commit 7 allowlisted source files locally; no push
[main 4f3a895] feat: implement Lab 5 managed training and release pipeline
 7 files changed, 551 insertions(+), 14 deletions(-)
 create mode 100644 lab5/scripts/inspect_existing.py
 create mode 100644 lab5/scripts/repair_serialization.py
 create mode 100644 lab5/scripts/resume_normal.py

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change 

RuntimeError: Normal repair หยุด: ส่ง summary และ child error ด้านบน ไม่ต้องเริ่มใหม่ครับ